# Advanced Topics

This notebook covers features for power users: programmatic species generation with `ListSpecies`, fractional stoichiometry, mathematical functions in rate expressions, and characteristic iteration.

In [ ]:
from mobspy import *

`ListSpecies` generates numbered species programmatically. This is useful when you need many identical species connected in a chain or grid.

In [ ]:
cells = ListSpecies(5)

# Chain reactions: each cell feeds into the next
for i in range(len(cells) - 1):
    cells[i] >> cells[i + 1][0.1]

cells[0](100)

S = Simulation(cells)
S.duration = 50
print(S.compile(verbose=True))

The compiled output shows 5 species (cells_0 through cells_4) and 4 reactions forming a linear chain.

Fractional stoichiometry handles yield coefficients less than 1. This is common in metabolic models where one molecule produces a fraction of another.

In [ ]:
A, B = BaseSpecies()

A >> 0.5 * B[1]

A(100)
B(0)

S2 = Simulation(A | B)
S2.duration = 20
print(S2.compile(verbose=True))

The stoichiometry of B in the product is 0.5. For every molecule of A consumed, half a molecule of B is produced (in continuous/deterministic simulation).

MobsPy provides mathematical functions for rate expressions: `ms_exp`, `ms_logn`, `ms_sin`, and others. These compile to COPASI-compatible function calls.

In [ ]:
from mobspy.modules.functions import ms_exp
from mobspy.modules.ode_operator import dt

C = BaseSpecies()

# Logistic-like growth with saturation via exponential
dt[C] += 10 / (1 + ms_exp(C / 100)) - 0.1 * C

C(50)

S3 = Simulation(C)
S3.duration = 100
S3.plot_data = False
S3.run()
print(S3.fres)

The `ms_exp(C / 100)` term creates a sigmoid saturation effect. Available functions include `ms_exp`, `ms_logn`, `ms_log10`, `ms_sin`, `ms_cos`, `ms_tan`, `ms_floor`, `ms_ceil`, `ms_abs`, and their hyperbolic variants.

For species with many characteristics, iterate with `get_characteristics()` and `.c()` to build reactions programmatically.

In [ ]:
Cell = BaseSpecies()
Cell.stage1
Cell.stage2
Cell.stage3
Cell.stage4
Cell.stage5

stages = sorted(Cell.get_characteristics())
print("Characteristics:", stages)

# Create transitions between consecutive stages
for i in range(len(stages) - 1):
    Cell.c(stages[i]) >> Cell.c(stages[i + 1])[0.1]

Cell.stage1(100)

S4 = Simulation(Cell)
S4.duration = 100
print(S4.compile(verbose=True))

This creates 4 transition reactions (stage1 to stage2, stage2 to stage3, etc.) without writing each one by hand. The `.c()` method selects a specific characteristic on a species.